In [ ]:
# ============================================================
# 정상기업 부실 확률 변화율 분석
# 2021_2024_PD_데이터.csv 기반
# ============================================================

import pandas as pd
import numpy as np
import os

PD_SAVE_DIR = r'15번. 우수모델 데이터'
PD_PATH     = os.path.join(PD_SAVE_DIR, "2021_2024_PD_데이터.csv")

YEAR_COL     = "회계년도"
COMPANY_COL  = "회사명"
ID_COL       = "사업자등록번호"
THRESHOLD    = 0.44
TARGET_YEARS = [2021, 2022, 2023, 2024]

# ── CSV 로드 ──────────────────────────────────────────────
pd_df = pd.read_csv(PD_PATH, encoding="utf-8-sig")
pd_df[YEAR_COL] = pd_df[YEAR_COL].astype(int)

print("=" * 65)
print(f"PD 데이터 로드 완료: {len(pd_df)}행")
print(f"연도 범위: {pd_df[YEAR_COL].min()} ~ {pd_df[YEAR_COL].max()}")
print(f"기업 수: {pd_df[ID_COL].nunique()}개")
print("=" * 65)


# ============================================================
# 대상 기업 필터링
# ============================================================

normal_2023 = pd_df[
    (pd_df[YEAR_COL] == 2023) &
    (pd_df["y_true"] == 0)
][[COMPANY_COL, ID_COL]].drop_duplicates()

target_ids = normal_2023[ID_COL].tolist()
print(f"2023년 정상 기업 수: {len(normal_2023)}개")

analysis_data = pd_df[
    pd_df[ID_COL].isin(target_ids) &
    pd_df[YEAR_COL].isin(TARGET_YEARS)
].copy()

year_count = (
    analysis_data.groupby(ID_COL)[YEAR_COL]
    .nunique()
    .reset_index()
    .rename(columns={YEAR_COL: "year_count"})
)
complete_ids  = year_count[year_count["year_count"] == 4][ID_COL].tolist()
analysis_data = analysis_data[analysis_data[ID_COL].isin(complete_ids)].copy()

print(f"2021~2024 모두 존재하는 기업 수: {len(complete_ids)}개")
print(f"분석 대상 행 수: {len(analysis_data)}행")
print("=" * 65)


# ============================================================
# 연도별 피벗
# ============================================================

pivot = analysis_data.pivot_table(
    index=[ID_COL, COMPANY_COL],
    columns=YEAR_COL,
    values="y_prob"
).reset_index()
pivot.columns.name = None
pivot = pivot.rename(columns={
    2021: "prob_2021",
    2022: "prob_2022",
    2023: "prob_2023",
    2024: "prob_2024",
})
pivot = pivot.dropna(
    subset=["prob_2021", "prob_2022", "prob_2023", "prob_2024"]
).reset_index(drop=True)


# ============================================================
# 변화율 계산
# ============================================================

def safe_change_rate(curr, prev):
    return np.where(
        prev.abs() < 1e-6,
        0.0,          # ★ NaN → 0으로 변경
        (curr - prev) / prev.abs() * 100
    )
pivot["change_2021_2022"] = safe_change_rate(
    pivot["prob_2022"], pivot["prob_2021"]
)
pivot["change_2022_2023"] = safe_change_rate(
    pivot["prob_2023"], pivot["prob_2022"]
)
pivot["change_2023_2024"] = safe_change_rate(
    pivot["prob_2024"], pivot["prob_2023"]
)


# ============================================================
# 예측 라벨
# ============================================================

for yr in TARGET_YEARS:
    pivot[f"pred_label_{yr}"] = (pivot[f"prob_{yr}"] >= THRESHOLD).astype(int)



result_df = pivot[[
    ID_COL, COMPANY_COL,
    "prob_2021", "prob_2022", "prob_2023", "prob_2024",
    "pred_label_2021", "pred_label_2022",
    "pred_label_2023", "pred_label_2024",
    "change_2021_2022", "change_2022_2023", "change_2023_2024",
]].copy()

for col in ["prob_2021", "prob_2022", "prob_2023", "prob_2024"]:
    result_df[col] = result_df[col].round(4)
for col in ["change_2021_2022", "change_2022_2023", "change_2023_2024"]:
    result_df[col] = result_df[col].round(2)



# ============================================================
# 정렬 (prob_2024 내림차순)
# ============================================================

result_df = result_df.sort_values(
    "prob_2024", ascending=False
).reset_index(drop=True)


# ============================================================
# 저장 (risk_signal 컬럼 제외)
# ============================================================

change_save_path = os.path.join("21번. 기업 PD 변화율/정상기업_부실확률_변화율.csv")
result_df[save_cols].to_csv(
    change_save_path, index=False, encoding="utf-8-sig"
)


# ============================================================
# 요약 출력
# ============================================================

print(f"\n{'='*65}")
print(f"정상기업 부실 확률 변화율 분석 완료")
print(f"  → {change_save_path}  ({len(result_df)}행)")



# ── 전체 상위 10개 ────────────────────────────────────────
print(f"\n  [전체 상위 10개 (prob_2024 높은 순)]")
print(result_df[[
    COMPANY_COL,
    "prob_2021", "prob_2022", "prob_2023", "prob_2024",
    "change_2021_2022", "change_2022_2023", "change_2023_2024"
]].head(10).to_string(index=False))

print("=" * 65)

PD 데이터 로드 완료: 15593행
연도 범위: 2021 ~ 2024
기업 수: 4436개
2023년 정상 기업 수: 3829개
2021~2024 모두 존재하는 기업 수: 3065개
분석 대상 행 수: 12260행

정상기업 부실 확률 변화율 분석 완료
  → 15번. 우수모델 데이터\정상기업_부실확률_변화율.csv  (3065행)

  [전체 상위 10개 (prob_2024 높은 순)]
           회사명  prob_2021  prob_2022  prob_2023  prob_2024  change_2021_2022  change_2022_2023  change_2023_2024
     주식회사디아이비즈     0.1476     0.5557     0.9546     0.9852            276.49             71.78              3.21
     주식회사엠지티앤씨     0.0025     0.8228     0.9521     0.9847          32812.00             15.71              3.42
       주식회사루덴스     0.0047     0.0804     0.9674     0.9803           1610.64           1103.23              1.33
        주식회사윙잇     0.0228     0.5176     0.9345     0.9794           2170.18             80.54              4.80
     메디랩코리아(주)     0.0441     0.0163     0.4622     0.9751            -63.04           2735.58            110.97
      (주)다름플러스     0.0010     0.1629     0.8556     0.9749          16190.00            425.23        